# Consensus Signal — Validation Notebook

**Components:** PT revision (35%), PT upside (15%), EPS growth (35%), Revenue growth (15%)  
**Pass criteria:** Sub-signal correlations ≥ 0.90. Composite correlation ≥ 0.85.

In [ ]:
import sys, os
os.chdir(os.path.expanduser("~/alpha-signal-v2"))
sys.path.insert(0, ".")

import pandas as pd
import numpy as np
from signals.consensus import _load_data, _compute_scores

## Step 1: Compute v2 and compare with v1

In [ ]:
stocks, consensus, fh, prices = _load_data()
v2 = _compute_scores(stocks, consensus, fh, prices)
print(f"v2: {len(v2)} stocks, {v2['consensus_signal'].notna().sum()} with signal")
for col in ["pt_upside", "pt_revision_1yr", "eps_growth", "revenue_growth"]:
    print(f"  {col}: {v2[col].notna().sum()} non-null")

In [ ]:
v1 = pd.read_csv(os.path.expanduser("~/alpha-signal/data/signals/consensus.csv"))
print(f"v1: {len(v1)} stocks")

merged = v2.merge(v1[["sid", "pt_upside", "pt_revision_1yr", "eps_growth_pct",
                       "revenue_growth_pct", "consensus_signal"]],
                  on="sid", how="inner", suffixes=("_v2", "_v1"))
print(f"Overlapping: {len(merged)}")

pairs = [
    ("pt_revision_1yr_v2", "pt_revision_1yr_v1", "PT Revision"),
    ("pt_upside_v2", "pt_upside_v1", "PT Upside"),
    ("eps_growth", "eps_growth_pct", "EPS Growth"),
    ("revenue_growth", "revenue_growth_pct", "Rev Growth"),
    ("consensus_signal_v2", "consensus_signal_v1", "Composite"),
]

print(f"\n{'Component':<16} {'Both':>8} {'Corr':>8} {'Mean|diff|':>10}")
print("-" * 46)
for v2c, v1c, label in pairs:
    both = merged[[v2c, v1c]].dropna()
    n = len(both)
    if n > 10:
        corr = both[v2c].corr(both[v1c])
        diff = (both[v2c] - both[v1c]).abs().mean()
        print(f"{label:<16} {n:>8} {corr:>8.4f} {diff:>10.4f}")
    else:
        print(f"{label:<16} {n:>8}   insufficient")

## Step 2: Save to DB (only if validation passed)

In [ ]:
# from signals.consensus import compute
# compute(dry_run=False)